In [2]:
# import Qdrant libraries
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import torch
from PIL import Image
import os
import json
# import huggingface libraries and the models we'll use
from transformers import CLIPProcessor, CLIPModel

c:\Users\payal\OneDrive\Desktop\image-rag-pipeline\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

coco_root = "./data/"  # CHANGE THIS
img_dir = os.path.join(coco_root, "images/")
ann_path = os.path.join(coco_root, "annotations", "captions_val2014.json")

# Load COCO captions json
with open(ann_path, "r") as f:
    coco = json.load(f)

# Map image_id to filename
image_id_to_filename = {img['id']: img['file_name'] for img in coco['images']}

# Get set of existing image filenames in img_dir
existing_filenames = set(os.listdir(img_dir))

# Filter to only image ids whose filenames exist in the folder
existing_image_ids = {img_id for img_id, fname in image_id_to_filename.items() if fname in existing_filenames}

# Build list of image paths and captions with filtering
image_caption_pairs = [
    {
        "image_path": os.path.join(img_dir, image_id_to_filename[ann["image_id"]]),
        "caption": ann["caption"]
    }
    for ann in coco['annotations']
    if ann["image_id"] in existing_image_ids
]

print(f"Number of image-caption pairs: {len(image_caption_pairs)}")

# Loop over your dataset (example with first 100)
embeddings = []
for idx, item in enumerate(image_caption_pairs[:100]):
    try:
        image = Image.open(item["image_path"]).convert("RGB")

        # process image (same as your original code, but moved inside loop)
        inputs = processor(images=image, return_tensors="pt").to(device)

        # forward pass
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)  # (1, 512)

        # normalize embedding
        image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

        # convert to numpy and save
        embedding_vector = image_embedding.cpu().numpy().squeeze()

        embeddings.append({
            "embedding": embedding_vector,
            "caption": item["caption"],
            "image_path": item["image_path"]
        })

        if idx % 10 == 0:
            print(f"Processed {idx} images")

    except Exception as e:
        print(f"Error processing {item['image_path']}: {e}")

print(f"Processed {len(embeddings)} embeddings in total.")


Number of image-caption pairs: 395
Processed 0 images
Processed 10 images
Processed 20 images
Processed 30 images
Processed 40 images
Processed 50 images
Processed 60 images
Processed 70 images
Processed 80 images
Processed 90 images
Processed 100 embeddings in total.
